In [52]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp as scipy_lse
from scipy.stats import norm
from scipy.integrate import quad
from scipy.special import expit
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.optimize import approx_fprime

Question 1

(b) an “overflow” error on a computer

In [11]:
def logsumexp(x):
    x = np.array(x, dtype=float)
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

# Demonstration
x = np.array([1000,50,40,10])

# Naive approach: overflow
naive = np.log(np.sum(np.exp(x)))
print(f"Naive: {naive}")

# Max trick
stable = logsumexp(x)
print(f"Max trick: {stable}")

Naive: inf
Max trick: 1000.0


C:\Users\HUAWEI\AppData\Local\Temp\ipykernel_3060\3025679522.py:10: RuntimeWarning: overflow encountered in exp
  naive = np.log(np.sum(np.exp(x)))


(c)

In [18]:
x = np.array([1000.0, 50.0, 40.0, 10.0])

# Check overflow
print(f"Our function: {logsumexp(x)}")
print(f"scipy:        {scipy_lse(x)}")

# Check underflow
x2 = np.array([-1000,-800,-900,-999.0])
print(f"\nOur function (underflow test): {logsumexp(x2)}")
print(f"scipy (underflow test):        {scipy_lse(x2)}")

Our function: 1000.0
scipy:        1000.0

Our function (underflow test): -800.0
scipy (underflow test):        -800.0


Question 2

(1)

In [25]:
def binomiallogit(beta, X=0.5, mu=0.5, sigma=2, pdf=norm.pdf):
    return expit(beta * X) * pdf(beta, loc=mu, scale=sigma)

(2)

In [28]:
true_val, _ = quad(binomiallogit, -np.inf, np.inf, epsabs=1e-14, epsrel=1e-14)
print(f"True value: {true_val}")

True value: 0.5514932726366428


(3)

In [29]:
np.random.seed(42)

for n in [20, 400]:
    draws = np.random.normal(0.5, 2, size=n)
    mc = np.mean(expit(draws * 0.5))
    print(f"Monte Carlo (n={n:3d}): {mc:.10f}, error: {mc - true_val:.2e}")

Monte Carlo (n= 20): 0.5163070144, error: -3.52e-02
Monte Carlo (n=400): 0.5571985673, error: 5.71e-03


(4)

In [30]:
from numpy.polynomial.hermite import hermgauss

mu, sigma, X = 0.5, 2, 0.5

for k in [4, 5, 7, 9, 12]:
    nodes, weights = hermgauss(k)
    beta = np.sqrt(2) * sigma * nodes + mu
    gh = np.sum(weights * expit(beta * X)) / np.sqrt(np.pi)
    print(f"GH (k={k:2d}): {gh:.10f}, error: {gh - true_val:.2e}")

GH (k= 4): 0.5513138768, error: -1.79e-04
GH (k= 5): 0.5515491037, error: 5.58e-05
GH (k= 7): 0.5514998647, error: 6.59e-06
GH (k= 9): 0.5514942243, error: 9.52e-07
GH (k=12): 0.5514932041, error: -6.86e-08


(5)

In [31]:
for k in [4, 5, 7, 9, 12]:
    nodes, weights = hermgauss(k)
    w_normalized = weights / np.sqrt(np.pi)
    print(f"GH (k={k:2d}): sum of weights = {np.sum(w_normalized):.10f}")

GH (k= 4): sum of weights = 1.0000000000
GH (k= 5): sum of weights = 1.0000000000
GH (k= 7): sum of weights = 1.0000000000
GH (k= 9): sum of weights = 1.0000000000
GH (k=12): sum of weights = 1.0000000000


(6)

In [32]:
from scipy.integrate import dblquad

mu1, mu2 = 0.5, 1.0
sig1, sig2 = 2.0, 1.0
X1, X2 = 0.5, 1.0

# True value via dblquad
def integrand_2d(b2, b1):
    return expit(b1*X1 + b2*X2) * norm.pdf(b1, mu1, sig1) * norm.pdf(b2, mu2, sig2)

true_2d, _ = dblquad(integrand_2d, -np.inf, np.inf, -np.inf, np.inf,
                      epsabs=1e-14, epsrel=1e-14)
print(f"True value (2D): {true_2d:.10f}")

# Monte Carlo
np.random.seed(42)
for n in [20, 400]:
    b1 = np.random.normal(mu1, sig1, size=n)
    b2 = np.random.normal(mu2, sig2, size=n)
    mc = np.mean(expit(b1*X1 + b2*X2))
    print(f"MC 2D (n={n:3d}): {mc:.10f}, error: {mc - true_2d:.2e}")

# Gauss-Hermite (tensor product)
for k in [4, 5, 7, 9, 12]:
    nodes, weights = hermgauss(k)
    beta1 = np.sqrt(2) * sig1 * nodes + mu1
    beta2 = np.sqrt(2) * sig2 * nodes + mu2
    gh = 0.0
    for i in range(k):
        for j in range(k):
            gh += weights[i] * weights[j] * expit(beta1[i]*X1 + beta2[j]*X2)
    gh /= np.pi  # 2D: divide by (sqrt(pi))^2 = pi
    print(f"GH 2D (k={k:2d}, pts={k**2:3d}): {gh:.10f}, error: {gh - true_2d:.2e}")

True value (2D): 0.7144838054
MC 2D (n= 20): 0.6513191925, error: -6.32e-02
MC 2D (n=400): 0.7219907386, error: 7.51e-03
GH 2D (k= 4, pts= 16): 0.7143713790, error: -1.12e-04
GH 2D (k= 5, pts= 25): 0.7144856148, error: 1.81e-06
GH 2D (k= 7, pts= 49): 0.7144830286, error: -7.77e-07
GH 2D (k= 9, pts= 81): 0.7144837635, error: -4.19e-08
GH 2D (k=12, pts=144): 0.7144838099, error: 4.46e-09


(7)

In [34]:
# === 1D Table ===
rows_1d = []
rows_1d.append({"Method": "quad (true)", "Points": "-", "Estimate": true_val, "Error": 0.0})

np.random.seed(42)
for n in [20, 400]:
    draws = np.random.normal(mu1, sig1, size=n)
    mc = np.mean(expit(draws * X1))
    rows_1d.append({"Method": f"MC (n={n})", "Points": n, "Estimate": mc, "Error": mc - true_val})

for k in [4, 5, 7, 9, 12]:
    nodes, weights = hermgauss(k)
    beta = np.sqrt(2) * sig1 * nodes + mu1
    gh = np.sum(weights * expit(beta * X1)) / np.sqrt(np.pi)
    rows_1d.append({"Method": f"GH (k={k})", "Points": k, "Estimate": gh, "Error": gh - true_val})

df_1d = pd.DataFrame(rows_1d)
print("=== 1D Integration ===")
print(df_1d.to_string(index=False))

# === 2D Table ===
rows_2d = []
rows_2d.append({"Method": "dblquad (true)", "Points": "-", "Estimate": true_2d, "Error": 0.0})

np.random.seed(42)
for n in [20, 400]:
    b1 = np.random.normal(mu1, sig1, size=n)
    b2 = np.random.normal(mu2, sig2, size=n)
    mc = np.mean(expit(b1*X1 + b2*X2))
    rows_2d.append({"Method": f"MC (n={n})", "Points": n, "Estimate": mc, "Error": mc - true_2d})

for k in [4, 5, 7, 9, 12]:
    nodes, weights = hermgauss(k)
    beta1 = np.sqrt(2) * sig1 * nodes + mu1
    beta2 = np.sqrt(2) * sig2 * nodes + mu2
    gh = 0.0
    for i in range(k):
        for j in range(k):
            gh += weights[i] * weights[j] * expit(beta1[i]*X1 + beta2[j]*X2)
    gh /= np.pi
    rows_2d.append({"Method": f"GH (k={k})", "Points": k**2, "Estimate": gh, "Error": gh - true_2d})

df_2d = pd.DataFrame(rows_2d)
print("\n=== 2D Integration ===")
print(df_2d.to_string(index=False))

=== 1D Integration ===
     Method Points  Estimate         Error
quad (true)      -  0.551493  0.000000e+00
  MC (n=20)     20  0.516307 -3.518626e-02
 MC (n=400)    400  0.557199  5.705295e-03
   GH (k=4)      4  0.551314 -1.793959e-04
   GH (k=5)      5  0.551549  5.583110e-05
   GH (k=7)      7  0.551500  6.592041e-06
   GH (k=9)      9  0.551494  9.516540e-07
  GH (k=12)     12  0.551493 -6.857120e-08

=== 2D Integration ===
        Method Points  Estimate         Error
dblquad (true)      -  0.714484  0.000000e+00
     MC (n=20)     20  0.651319 -6.316461e-02
    MC (n=400)    400  0.721991  7.506933e-03
      GH (k=4)     16  0.714371 -1.124264e-04
      GH (k=5)     25  0.714486  1.809372e-06
      GH (k=7)     49  0.714483 -7.767488e-07
      GH (k=9)     81  0.714484 -4.193391e-08
     GH (k=12)    144  0.714484  4.463599e-09


Question 3

(1)

In [36]:
df = pd.read_csv('pset1_data.csv')

In [37]:
df

,Unnamed: 0,individual_id_i,apt_id_j,family_size_i,logsqfeet_j,bath_j,outdoor_j,y_ij
0,0,0,0,3,5.759642,1,1.482007,0
1,1,0,1,3,6.377517,2,1.178354,0
2,2,0,2,3,6.043631,2,2.453078,0
3,3,0,3,3,6.430696,2,4.675143,1
4,4,0,4,3,5.270265,1,0.700370,0
...,...,...,...,...,...,...,...,...
199995,209994,9999,15,1,6.062593,2,3.085974,0
199996,209995,9999,16,1,6.132612,1,3.871953,0
199997,209996,9999,17,1,6.241101,1,3.800217,0
199998,209997,9999,18,1,6.146578,2,4.880468,0


In [46]:
# Construct variables
N = df['individual_id_i'].nunique()
J = df['apt_id_j'].nunique()

logsqft = df['logsqfeet_j'].values.reshape(N, J)
bath = df['bath_j'].values.reshape(N, J)
bath_fam = (df['bath_j'] * df['family_size_i']).values.reshape(N, J)
outdoor = df['outdoor_j'].values.reshape(N, J)
y = df['y_ij'].values.reshape(N, J)

In [47]:
def log_likelihood(params):
    b0, b1, b2, b3, gamma = params
    V = b0 + b1*logsqft + b2*bath + b3*bath_fam + gamma*outdoor  # (N, J)
    # Add outside option with V=0
    V_full = np.hstack([np.zeros((N, 1)), V])  # (N, J+1)
    log_denom = logsumexp(V_full, axis=1)
    log_num = np.sum(y * V, axis=1)
    return -np.sum(log_num - log_denom)

In [48]:
x0 = np.zeros(5)
res = minimize(log_likelihood, x0, method='BFGS')

labels = ['beta0', 'beta1', 'beta2', 'beta3', 'gamma']
print("Plain Logit MLE Results:")
for name, val in zip(labels, res.x):
    print(f"  {name:6s} = {val:.6f}")
print(f"  Log-likelihood = {-res.fun:.4f}")

Plain Logit MLE Results:
  beta0  = -4.813509
  beta1  = -0.109368
  beta2  = 0.767902
  beta3  = 0.042020
  gamma  = 0.809400
  Log-likelihood = -24640.8475


(b)

In [49]:
K = 12  # number of quadrature nodes

nodes, weights = hermgauss(K)
w_norm = weights / np.sqrt(np.pi)

In [50]:
def mixed_log_likelihood(params):
    b0, b1, b2, b3, gamma_bar, log_sigma = params
    sigma = np.exp(log_sigma)  # ensure sigma > 0

    ll = np.zeros(N)
    for s in range(K):
        gamma_s = np.sqrt(2) * sigma * nodes[s] + gamma_bar
        V = b0 + b1*logsqft + b2*bath + b3*bath_fam + gamma_s*outdoor
        V_full = np.hstack([np.zeros((N, 1)), V])
        log_denom = logsumexp(V_full, axis=1)
        log_num = np.sum(y * V, axis=1)
        # P(choice | gamma_s) for each individual
        ll += w_norm[s] * np.exp(log_num - log_denom)

    return -np.sum(np.log(ll))

In [51]:
# Use plain logit estimates as starting values
x0 = np.array([res.x[0], res.x[1], res.x[2], res.x[3], res.x[4], np.log(1.0)])
res2 = minimize(mixed_log_likelihood, x0, method='Nelder-Mead',
                options={'maxiter': 100000, 'xatol': 1e-8, 'fatol': 1e-8})

labels = ['beta0', 'beta1', 'beta2', 'beta3', 'gamma_bar', 'sigma']
vals = list(res2.x[:5]) + [np.exp(res2.x[5])]
print("Mixed Logit MSL Results:")
for name, val in zip(labels, vals):
    print(f"  {name:10s} = {val:.6f}")
print(f"  Log-likelihood = {-res2.fun:.4f}")

Mixed Logit MSL Results:
  beta0      = -4.300304
  beta1      = 0.468251
  beta2      = 0.328670
  beta3      = 0.071623
  gamma_bar  = 1.071004
  sigma      = 2.148281
  Log-likelihood = -23812.9849


(3)

In [53]:

# Standard errors for plain logit
H1 = np.zeros((5, 5))
eps = 1e-5
for i in range(5):
    def grad_i(params):
        return approx_fprime(params, log_likelihood, eps)[i]
    H1[i] = approx_fprime(res.x, grad_i, eps)
se1 = np.sqrt(np.diag(np.linalg.inv(H1)))

labels1 = ['beta0', 'beta1', 'beta2', 'beta3', 'gamma']
print("Plain Logit Standard Errors:")
for name, val, se in zip(labels1, res.x, se1):
    print(f"  {name:6s} = {val:.6f}  (SE = {se:.6f})")

# Standard errors for mixed logit
n_params = 6
H2 = np.zeros((n_params, n_params))
for i in range(n_params):
    def grad_i(params):
        return approx_fprime(params, mixed_log_likelihood, eps)[i]
    H2[i] = approx_fprime(res2.x, grad_i, eps)
se2_raw = np.sqrt(np.diag(np.linalg.inv(H2)))

# Delta method for sigma = exp(log_sigma)
sigma_hat = np.exp(res2.x[5])
se2 = list(se2_raw[:5]) + [sigma_hat * se2_raw[5]]

labels2 = ['beta0', 'beta1', 'beta2', 'beta3', 'gamma_bar', 'sigma']
vals2 = list(res2.x[:5]) + [sigma_hat]
print("\nMixed Logit Standard Errors:")
for name, val, se in zip(labels2, vals2, se2):
    print(f"  {name:10s} = {val:.6f}  (SE = {se:.6f})")

Plain Logit Standard Errors:
  beta0  = -4.813509  (SE = 0.284183)
  beta1  = -0.109368  (SE = 0.042234)
  beta2  = 0.767902  (SE = 0.043179)
  beta3  = 0.042020  (SE = 0.015958)
  gamma  = 0.809400  (SE = 0.012333)

Mixed Logit Standard Errors:
  beta0      = -4.300304  (SE = 0.258939)
  beta1      = 0.468251  (SE = 0.043299)
  beta2      = 0.328670  (SE = 0.073124)
  beta3      = 0.071623  (SE = 0.032602)
  gamma_bar  = 1.071004  (SE = 0.031311)
  sigma      = 2.148281  (SE = 0.062968)
